In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random

In [ ]:
# COLOR MAP
CITYSCAPES_PALETTE = np.array([(0, 0, 0), (111, 74, 0), (81, 0, 81), (128, 64, 128), (244, 35, 232), (250, 170, 160), (230, 150, 140), (70, 70, 70), 
            (102, 102, 156), (190, 153, 153), (180, 165, 180), (150, 100, 100), (150, 120, 90), (153, 153, 153), (250, 170, 30), (220, 220, 0), (107, 142, 35), 
            (152, 251, 152), (70, 130, 180), (220, 20, 60), (255, 0, 0), ( 0, 0, 142), ( 0, 0, 70), (0, 60, 100), (0, 0, 90), (0, 0, 110), (0, 80, 100), (0, 0, 230), 
            (119, 11, 32), (0, 0, 142)], dtype = np.int32)

In [ ]:
def rgb_to_class(mask_arr):
    #to find the closesr palette that to map the label pixels
    diff = mask_arr[:, :, np.newaxis, :] - CITYSCAPES_PALETTE
    dist = np.sum(diff ** 2, axis=-1)
    class_mask = np.argmin(dist, axis=-1) 
    return class_mask


class SegmentationDataset(Dataset):
    def __init__(self, root_dir, image_size=(256,96), train=True):
        self.img_dir = os.path.join(root_dir, 'img')
        self.label_dir = os.path.join(root_dir, 'label')
        self.image_size = image_size
        self.train = train
        self.images = sorted([f for f in os.listdir(self.img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        image = Image.open(os.path.join(self.img_dir, img_name)).convert('RGB')

        # Load mask as RGB so we can match colors
        label_img = Image.open(os.path.join(self.label_dir, img_name)).convert('RGB')

        # Resize
        image = image.resize(self.image_size, Image.BILINEAR)
        label_img = label_img.resize(self.image_size, Image.NEAREST)

        # Convert image to tensor
        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        # Map noisy RGB mask to clean class IDs (0-19)
        label_arr = np.array(label_img, dtype=np.int32)
        class_mask = rgb_to_class(label_arr)
        label = torch.tensor(class_mask, dtype=torch.long)

        # Data Augmentation
        if self.train and random.random() > 0.5:
            image = TF.hflip(image)
            label = TF.hflip(label)

        return image, label


In [ ]:
#device = "mps" if torch.backends.mps.is_available() else "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

train_dataset = SegmentationDataset('/kaggle/input/cityscapes/train', image_size=(256,96), train=True)
val_dataset   = SegmentationDataset('/kaggle/input/cityscapes/val',   image_size=(256,96), train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:

class DoubleConv(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.doubleConv=nn.Sequential(
                nn.Conv2d(input_dim,output_dim,kernel_size=3,padding=1),
                nn.BatchNorm2d(output_dim),
                nn.ReLU(),
                nn.Conv2d(output_dim,output_dim,kernel_size=3,padding=1),
                nn.BatchNorm2d(output_dim),
                nn.ReLU())
    def forward(self,x):
        return self.doubleConv(x)

class Encoder(nn.Module):
    def __init__(self, input_dim, features):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        for feature in features:
            self.encoders.append(DoubleConv(input_dim, feature))
            input_dim = feature
    def forward(self, x):
        skip_connections =[]
        for block in self.encoders:
            x = block(x)
            skip_connections.append(x)
            x = self.pool(x)
        return x, skip_connections

In [ ]:

class BottleNeck(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.conv=DoubleConv(input_dim,output_dim)
    def forward(self,x):
        return self.conv(x)

class Decoder(nn.Module):
    def __init__(self,input_channels,skip_conn,out_channels):
        super().__init__()
        self.up=nn.ConvTranspose2d(input_channels,skip_conn,kernel_size=2,stride=2)
        self.conv=DoubleConv(skip_conn*2,out_channels)
    def forward(self,x,skip):
        x=self.up(x)
        diff_x=skip.size()[2] - x.size()[2]
        diff_y=skip.size()[3] - x.size()[3]
        x=F.pad(x,[diff_x//2,diff_x-diff_x//2, diff_y//2,diff_y-diff_y//2])
        x=torch.cat([skip,x],dim=1)
        return self.conv(x)

In [ ]:
class Unet(nn.Module):
    def __init__(self,input_channels=3,num_classes=30,features=[64,128,256,512]):
        super().__init__()
        self.encoder=Encoder(input_channels,features)
        self.bottleNeck=BottleNeck(features[-1],features[-1]*2)
        self.decoders=nn.ModuleList()
        rev_features=features[::-1]
        in_ch=features[-1]*2
        for skip_conn in rev_features:
            out_ch=skip_conn
            self.decoders.append(Decoder(in_ch,skip_conn,out_ch))
            in_ch=out_ch
        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)
    def forward(self,x):
        x,skips=self.encoder(x)
        x=self.bottleNeck(x)
        skips=skips[::-1]
        for i,decoder in enumerate(self.decoders):
            x=decoder(x,skips[i])
        return self.final_conv(x)

In [ ]:
model = Unet(num_classes=30).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=19)
torch.backends.cudnn.benchmark = False

In [3]:
epochs = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_pixels = 0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        with torch.no_grad():
            preds = torch.argmax(outputs, dim=1)
            valid_mask = masks != 29
            train_correct += (preds[valid_mask] == masks[valid_mask]).sum().item()
            train_pixels += valid_mask.sum().item()

    train_loss /= len(train_dataset)
    train_acc = train_correct / train_pixels if train_pixels > 0 else 0

    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Train Acc {train_acc:.4f}")

Using device: cuda
Epoch 1: Train Loss 1.5801, Train Acc 0.6643
Epoch 2: Train Loss 1.0203, Train Acc 0.7500
Epoch 3: Train Loss 0.8693, Train Acc 0.7704
Epoch 4: Train Loss 0.7918, Train Acc 0.7826
Epoch 5: Train Loss 0.7398, Train Acc 0.7930
Epoch 6: Train Loss 0.7021, Train Acc 0.7994
Epoch 7: Train Loss 0.6733, Train Acc 0.8051
Epoch 8: Train Loss 0.6525, Train Acc 0.8092
Epoch 9: Train Loss 0.6276, Train Acc 0.8152
Epoch 10: Train Loss 0.6148, Train Acc 0.8169
Epoch 11: Train Loss 0.5951, Train Acc 0.8223
Epoch 12: Train Loss 0.5848, Train Acc 0.8243
Epoch 13: Train Loss 0.5714, Train Acc 0.8275
Epoch 14: Train Loss 0.5583, Train Acc 0.8305
Epoch 15: Train Loss 0.5482, Train Acc 0.8330
Epoch 16: Train Loss 0.5375, Train Acc 0.8359
Epoch 17: Train Loss 0.5311, Train Acc 0.8372
Epoch 18: Train Loss 0.5162, Train Acc 0.8412
Epoch 19: Train Loss 0.5097, Train Acc 0.8426
Epoch 20: Train Loss 0.4998, Train Acc 0.8455


In [ ]:
path="unetmodel.pth"
torch.save(model.state_dict(),path)

In [4]:
def predict_unet(model_path, image_path, features=[64,128,256,512], num_classes=30,
                 device=None, image_size=(96,256), palette=CITYSCAPES_PALETTE):

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    
    model = Unet(num_classes=num_classes, features=features)
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()


    image = Image.open(image_path).convert('RGB')

    image = image.resize((image_size[1], image_size[0]), Image.BILINEAR)

    input_tensor = TF.to_tensor(image)
    input_tensor = TF.normalize(input_tensor,
                                 mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
    input_tensor = input_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)               
        pred_class = torch.argmax(logits, dim=1)   

    mask_np = pred_class.squeeze().cpu().numpy().astype(np.uint8)
    mask_pil = Image.fromarray(mask_np).convert('P')

    if palette is not None:
        flat_palette = palette.flatten().tolist()
        flat_palette += [0] * (768 - len(flat_palette))

        mask_pil.putpalette(flat_palette)

    return mask_pil, logits.cpu()



In [ ]:
mask2, _ = predict_unet(
    model_path='unetmodel.pth',
    image_path='/kaggle/input/datasets/bot00096601/vallll/val98.png',
    num_classes=30, 
    palette=CITYSCAPES_PALETTE
)
mask2.save('coloured_mask2.png')

In [ ]:
mask1, _ = predict_unet(
    model_path='unetmodel.pth',
    image_path='/kaggle/input/datasets/bot00096601/vallll/val19.png',
    num_classes=30, 
    palette=CITYSCAPES_PALETTE
)
mask1.save('coloured_mask1.png')